In [14]:
import os
import re
import json
from datetime import datetime
import pandas as pd
import numpy as np
from dateutil import parser as date_parser

In [15]:
df = pd.read_csv("data/books_raw.csv", encoding="utf-8")
print("Shape:", df.shape)
df.head()

Shape: (1000, 4)


,Title,Price,Availability,Rating
0,A Light in the Attic,Â£51.77,In stock,Three
1,Tipping the Velvet,Â£53.74,In stock,One
2,Soumission,Â£50.10,In stock,One
3,Sharp Objects,Â£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five


In [16]:
# Remove characters like Â from all text

df = df.replace(r"Â", "", regex=True)
df

,Title,Price,Availability,Rating
0,A Light in the Attic,£51.77,In stock,Three
1,Tipping the Velvet,£53.74,In stock,One
2,Soumission,£50.10,In stock,One
3,Sharp Objects,£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five
...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,£55.53,In stock,One
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",£57.06,In stock,Four
997,A Spy's Devotion (The Regency Spies of London #1),£16.97,In stock,Five
998,1st to Die (Women's Murder Club #1),£53.98,In stock,One


In [17]:
# Strip extra spaces from title

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()
df.head()

,Title,Price,Availability,Rating
0,A Light in the Attic,£51.77,In stock,Three
1,Tipping the Velvet,£53.74,In stock,One
2,Soumission,£50.10,In stock,One
3,Sharp Objects,£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five


In [18]:
# Remove currency symbols and convert to float

df["Price"] = df["Price"].str.replace(r"[^0-9\.]", "", regex=True)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Price"].head()

0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
Name: Price, dtype: float64

In [19]:
# Map the ratings with numbers

rating_map = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}
df["Rating"] = df["Rating"].map(rating_map)
df["Rating"].head()

0    3
1    1
2    1
3    4
4    5
Name: Rating, dtype: int64

In [20]:
# remove newlines, multiple spaces

df["Title"] = df["Title"].str.replace(r"\s+", " ", regex=True)

df

,Title,Price,Availability,Rating
0,A Light in the Attic,51.77,In stock,3
1,Tipping the Velvet,53.74,In stock,1
2,Soumission,50.10,In stock,1
3,Sharp Objects,47.82,In stock,4
4,Sapiens: A Brief History of Humankind,54.23,In stock,5
...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,55.53,In stock,1
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,In stock,4
997,A Spy's Devotion (The Regency Spies of London #1),16.97,In stock,5
998,1st to Die (Women's Murder Club #1),53.98,In stock,1


In [22]:
# Lowercase & remove non-alphanumeric (except # and @)

df["Title_clean"] = df["Title"].str.lower().str.replace(r"[^a-z0-9\s#@]", "", regex=True)
df[["Title", "Title_clean"]]

,Title,Title_clean
0,A Light in the Attic,a light in the attic
1,Tipping the Velvet,tipping the velvet
2,Soumission,soumission
3,Sharp Objects,sharp objects
4,Sapiens: A Brief History of Humankind,sapiens a brief history of humankind
...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,alice in wonderland alices adventures in wonde...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",ajin demihuman volume 1 ajin demihuman #1
997,A Spy's Devotion (The Regency Spies of London #1),a spys devotion the regency spies of london #1
998,1st to Die (Women's Murder Club #1),1st to die womens murder club #1


In [23]:
#Extract hashtags, mentions, keywords etc

def extract_hashtags(text):
    return re.findall(r"#(\w+)", text)

def extract_mentions(text):
    return re.findall(r"@(\w+)", text)

def extract_keywords(text):
    words = text.split()
    stopwords = {"the","and","of","in","a","to","for","on"}
    return [w for w in words if w not in stopwords and len(w) > 2]

df["Hashtags"] = df["Title_clean"].apply(extract_hashtags)
df["Mentions"] = df["Title_clean"].apply(extract_mentions)
df["Keywords"] = df["Title_clean"].apply(extract_keywords)

df[["Title_clean","Hashtags","Mentions","Keywords"]].head(10)


,Title_clean,Hashtags,Mentions,Keywords
0,a light in the attic,[],[],"[light, attic]"
1,tipping the velvet,[],[],"[tipping, velvet]"
2,soumission,[],[],[soumission]
3,sharp objects,[],[],"[sharp, objects]"
4,sapiens a brief history of humankind,[],[],"[sapiens, brief, history, humankind]"
5,the requiem red,[],[],"[requiem, red]"
6,the dirty little secrets of getting your dream...,[],[],"[dirty, little, secrets, getting, your, dream,..."
7,the coming woman a novel based on the life of ...,[],[],"[coming, woman, novel, based, life, infamous, ..."
8,the boys in the boat nine americans and their ...,[],[],"[boys, boat, nine, americans, their, epic, que..."
9,the black maria,[],[],"[black, maria]"


In [24]:
# Fill missing titles

df["Title_clean"] = df["Title_clean"].fillna("unknown_title")
df.head()

,Title,Price,Availability,Rating,Title_clean,Hashtags,Mentions,Keywords
0,A Light in the Attic,51.77,In stock,3,a light in the attic,[],[],"[light, attic]"
1,Tipping the Velvet,53.74,In stock,1,tipping the velvet,[],[],"[tipping, velvet]"
2,Soumission,50.10,In stock,1,soumission,[],[],[soumission]
3,Sharp Objects,47.82,In stock,4,sharp objects,[],[],"[sharp, objects]"
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,sapiens a brief history of humankind,[],[],"[sapiens, brief, history, humankind]"


In [26]:
# Fill missing prices with median price

df["Price"] = df["Price"].fillna(df["Price"].median())
df.head()

,Title,Price,Availability,Rating,Title_clean,Hashtags,Mentions,Keywords
0,A Light in the Attic,51.77,In stock,3,a light in the attic,[],[],"[light, attic]"
1,Tipping the Velvet,53.74,In stock,1,tipping the velvet,[],[],"[tipping, velvet]"
2,Soumission,50.10,In stock,1,soumission,[],[],[soumission]
3,Sharp Objects,47.82,In stock,4,sharp objects,[],[],"[sharp, objects]"
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,sapiens a brief history of humankind,[],[],"[sapiens, brief, history, humankind]"


In [28]:
# Convert list obj to strings (if not)

df = df.astype(str)

# drop duplicates
df = df.drop_duplicates()

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Title         1000 non-null   object
 1   Price         1000 non-null   object
 2   Availability  1000 non-null   object
 3   Rating        1000 non-null   object
 4   Title_clean   1000 non-null   object
 5   Hashtags      1000 non-null   object
 6   Mentions      1000 non-null   object
 7   Keywords      1000 non-null   object
dtypes: object(8)
memory usage: 62.6+ KB


In [31]:
#Cleaned DF

df.head(10)

,Title,Price,Availability,Rating,Title_clean,Hashtags,Mentions,Keywords
0,A Light in the Attic,51.77,In stock,3,a light in the attic,[],[],"['light', 'attic']"
1,Tipping the Velvet,53.74,In stock,1,tipping the velvet,[],[],"['tipping', 'velvet']"
2,Soumission,50.1,In stock,1,soumission,[],[],['soumission']
3,Sharp Objects,47.82,In stock,4,sharp objects,[],[],"['sharp', 'objects']"
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,sapiens a brief history of humankind,[],[],"['sapiens', 'brief', 'history', 'humankind']"
5,The Requiem Red,22.65,In stock,1,the requiem red,[],[],"['requiem', 'red']"
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4,the dirty little secrets of getting your dream...,[],[],"['dirty', 'little', 'secrets', 'getting', 'you..."
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,3,the coming woman a novel based on the life of ...,[],[],"['coming', 'woman', 'novel', 'based', 'life', ..."
8,The Boys in the Boat: Nine Americans and Their...,22.6,In stock,4,the boys in the boat nine americans and their ...,[],[],"['boys', 'boat', 'nine', 'americans', 'their',..."
9,The Black Maria,52.15,In stock,1,the black maria,[],[],"['black', 'maria']"


In [32]:
df.to_csv("data/cleaned_books.csv", index=False, encoding="utf-8")
print("Cleaned data saved to data/cleaned_books.csv")

Cleaned data saved to data/cleaned_books.csv
